# Phase 1 — Hybrid ReAct Pipeline: Interactive Demo

**Author:** Principal AI Architect, Infosys Procurement Agent  
**Reference:** `KnowledgeBase/project_scaffold/ai_models/comparasion_scoring_model/phase1_hybrid_react.md`  
**Status:** Production baseline — fixed-weight heuristic scorer

---

## Architecture Overview

The Phase 1 pipeline is a **two-stage hybrid**: a deterministic quantitative subgraph computes raw TCO/feasibility signals using NumPy/Pandas; a GPT-4o-backed ReAct agent (implemented in LangGraph) then augments each feasible offering with a qualitative score derived from supplier history, ESG reports, and certification checks. A final fixed-weight aggregator collapses both signals into a single ranked list.

```
Beckn on_search payload
        │
        ▼
  Pydantic validator          ← DiscoverOffering schema
        │
        ▼
  ┌─────────────────────────────────────────┐
  │   QUANT SUBGRAPH (deterministic)        │
  │  TCO = P·q·(1−δ(q)) + C_ship           │
  │        + C_risk(σ,τ) + C_ops           │
  │  Hard filter: τ≥0 ∧ TCO≤B ∧ ¬blacklist │
  └─────────────────────────────────────────┘
        │
        ▼
  ┌─────────────────────────────────────────┐
  │   ReAct AGENT LOOP (qualitative)        │
  │  Thought → Action → Observation × k    │
  │  Tools: history / ESG / certification  │
  └─────────────────────────────────────────┘
        │
        ▼
  Fixed-weight aggregator
  S_i = 0.40·x_tco + 0.25·x_τ + 0.20·x_qual + 0.15·x_risk
        │
        ▼
  POST /score  →  selected DiscoverOffering
```

**This notebook is fully self-contained.** All tool calls in the ReAct loop return mocked responses — no API key is required.

---

## Notebook Structure

| Cell | Stage | Role |
|------|-------|------|
| 1 | Setup | Imports and Pydantic models |
| 2 | Data | Mock offerings + buyer intent |
| 3 | QUANT | TCO, discounts, risk, hard filter |
| 4 | Normalise | Session-level min-max scaling |
| 5 | ReAct | LangGraph qualitative agent |
| 6 | Score | Fixed-weight aggregation + ranking |

In [ ]:
# Cell 1 — Imports and Pydantic models
from __future__ import annotations

import json
import math
import textwrap
from importlib.metadata import version as _pkg_version
from typing import Any, Literal, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, field_validator

# LangGraph / LangChain (mock-compatible — no LLM calls made)
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Shared Pydantic models (mirror shared/models.py) ────────────────────────

class BudgetConstraints(BaseModel):
    max: float
    min: float = 0.0

class BecknIntent(BaseModel):
    item_name: str
    quantity: int
    delivery_timeline: int          # hours (not ISO 8601)
    location_coordinates: str       # "lat,lon" decimal
    budget_constraints: BudgetConstraints

    @field_validator("location_coordinates")
    @classmethod
    def _validate_latlon(cls, v: str) -> str:
        parts = v.split(",")
        assert len(parts) == 2, "location_coordinates must be 'lat,lon'"
        float(parts[0]); float(parts[1])
        return v

class DiscountTier(BaseModel):
    """Volume discount tier: applies rate `discount` when qty >= `min_qty`."""
    min_qty: int
    max_qty: int        # exclusive upper bound; use math.inf for the last tier
    discount: float     # fractional (0.05 = 5 %)

class DiscoverOffering(BaseModel):
    offering_id: str
    supplier_name: str
    price_value: float              # unit price in INR
    delivery_time_hours: int        # promised delivery window (hours)
    shipping_cost: float            # flat shipping in INR
    ops_overhead: float             # estimated operational cost in INR
    reliability_sigma: float        # std-dev of past delivery lateness (hours)
    blacklisted: bool = False
    certified: bool = True          # ISO/BIS certification flag
    discount_tiers: list[DiscountTier] = Field(default_factory=list)
    esg_score: Optional[float] = None   # optional pre-fetched ESG score [0,1]

print("✓ Imports OK — LangGraph:", _pkg_version("langgraph"),
      "| Pydantic:", __import__('pydantic').__version__)

## Cell 2 — Mock Data

We simulate a Beckn `on_search` callback that delivered **5 supplier offerings** for `300 × Cat6 UTP cable` to the `comparative-scoring` microservice (`:8003`).  

The buyer's `BecknIntent` encodes:
- **Quantity:** 300 units  
- **Deadline:** 72 hours (3 days)  
- **Budget cap:** ₹1,50,000  
- **Location:** Mumbai (19.0760°N, 72.8777°E)

Offering layout:

| ID | Supplier | Unit Price | Ship | σ | Delivery | Tiers |
|----|----------|-----------|------|---|----------|-------|
| O1 | TechWire | ₹420 | ₹2,000 | 4 h | 60 h | 5%@100, 10%@300 |
| O2 | CableMart | ₹395 | ₹2,500 | 8 h | 48 h | 8%@200 |
| O3 | InfraNet | ₹445 | ₹1,800 | 3 h | 36 h | 7%@150 |
| O4 | QuickLink | ₹380 | ₹3,200 | 12 h | 96 h | 4%@50 | ← fails deadline |
| O5 | GlobalCables | ₹460 | ₹1,500 | 2 h | 24 h | 12%@250 |

In [ ]:
# Cell 2 — Mock DiscoverOffering list + BecknIntent

INTENT = BecknIntent(
    item_name="Cat6 UTP Cable",
    quantity=300,
    delivery_timeline=72,           # buyer needs delivery within 72 h
    location_coordinates="19.0760,72.8777",
    budget_constraints=BudgetConstraints(max=150_000, min=0)
)

OFFERINGS: list[DiscoverOffering] = [
    DiscoverOffering(
        offering_id="O1", supplier_name="TechWire Pvt Ltd",
        price_value=420.0, delivery_time_hours=60,
        shipping_cost=2000.0, ops_overhead=500.0,
        reliability_sigma=4.0, certified=True,
        discount_tiers=[
            DiscountTier(min_qty=100, max_qty=300, discount=0.05),
            DiscountTier(min_qty=300, max_qty=100_000, discount=0.10),
        ]
    ),
    DiscoverOffering(
        offering_id="O2", supplier_name="CableMart India",
        price_value=395.0, delivery_time_hours=48,
        shipping_cost=2500.0, ops_overhead=400.0,
        reliability_sigma=8.0, certified=True,
        discount_tiers=[
            DiscountTier(min_qty=200, max_qty=100_000, discount=0.08),
        ]
    ),
    DiscoverOffering(
        offering_id="O3", supplier_name="InfraNet Solutions",
        price_value=445.0, delivery_time_hours=36,
        shipping_cost=1800.0, ops_overhead=300.0,
        reliability_sigma=3.0, certified=True,
        discount_tiers=[
            DiscountTier(min_qty=150, max_qty=100_000, discount=0.07),
        ]
    ),
    DiscoverOffering(
        offering_id="O4", supplier_name="QuickLink Distributors",
        price_value=380.0, delivery_time_hours=96,   # ← violates 72-h deadline
        shipping_cost=3200.0, ops_overhead=600.0,
        reliability_sigma=12.0, certified=False,     # ← no cert
        discount_tiers=[
            DiscountTier(min_qty=50, max_qty=100_000, discount=0.04),
        ]
    ),
    DiscoverOffering(
        offering_id="O5", supplier_name="GlobalCables Corp",
        price_value=460.0, delivery_time_hours=24,
        shipping_cost=1500.0, ops_overhead=200.0,
        reliability_sigma=2.0, certified=True,
        discount_tiers=[
            DiscountTier(min_qty=250, max_qty=100_000, discount=0.12),
        ]
    ),
]

print(f"Buyer intent : {INTENT.item_name}  qty={INTENT.quantity}  "
      f"deadline={INTENT.delivery_timeline}h  budget=₹{INTENT.budget_constraints.max:,.0f}")
print(f"Offerings    : {len(OFFERINGS)} suppliers received from Beckn on_search callback")
for o in OFFERINGS:
    print(f"  {o.offering_id}  {o.supplier_name:<28}  ₹{o.price_value}/unit  "
          f"{o.delivery_time_hours}h  σ={o.reliability_sigma}h  "
          f"cert={'✓' if o.certified else '✗'}")

## Cell 3 — Quantitative Subgraph: TCO, Volume Discounts & Hard Filter

### TCO Formula

For offering $i$, quantity $q$, deadline $D$:

$$\text{TCO}_i = P_i \cdot q \cdot (1 - \delta_i(q)) + C_{\text{ship},i} + C_{\text{risk},i}(\sigma_i, \tau_i) + C_{\text{ops},i}$$

**Volume discount step-function** (implemented via `numpy.digitize`):

$$\delta_i(q) = \sum_k d_{i,k} \cdot \mathbf{1}\bigl[q \in [q_{i,k},\, q_{i,k+1})\bigr]$$

**Delivery slack:**

$$\tau_i = D_{\text{deadline}} - D_{\text{promised},i} \quad (\text{hours})$$

**Risk premium** (penalises unreliable suppliers and late deliveries):

$$C_{\text{risk},i} = \alpha \cdot \sigma_i \cdot \max(0, -\tau_i) + \beta \cdot \sigma_i^2$$

where $\alpha = 150$ (INR per hour of lateness per σ-unit) and $\beta = 50$ (INR per σ²).  
When $\tau_i \geq 0$ the penalty collapses to $\beta \cdot \sigma_i^2$ (variance cost only).

**Hard constraint filter** — offering is infeasible if **any** of:

$$\mathcal{S}_{\text{feasible}} = \{i \mid \tau_i \geq 0 \;\land\; \text{TCO}_i \leq B \;\land\; \text{blacklist}(i) = \text{false}\}$$

Note: certification is used as a qualitative signal in the ReAct loop (Cell 5), not as a hard filter here, to match the production architecture.

In [ ]:
# Cell 3 — TCO computation, volume discounts via np.digitize, hard filter

ALPHA = 150.0   # INR / (hour × σ-unit)  — lateness penalty weight
BETA  =  50.0   # INR / σ²               — variance (reliability) cost


def volume_discount(offering: DiscoverOffering, qty: int) -> float:
    """Return δ_i(q) using np.digitize on the tier breakpoints."""
    tiers = offering.discount_tiers
    if not tiers:
        return 0.0
    # Build breakpoints array: [min_qty_tier_0, min_qty_tier_1, ...]
    breakpoints = np.array([t.min_qty for t in tiers], dtype=float)
    # digitize returns the index of the bin that qty falls into.
    # bins are right-open: breakpoints[k] <= qty < breakpoints[k+1]
    idx = np.digitize(qty, breakpoints, right=False) - 1
    if idx < 0:
        return 0.0          # qty below first tier
    idx = min(idx, len(tiers) - 1)
    tier = tiers[idx]
    # Verify qty is within tier's explicit max bound
    if qty < tier.max_qty:
        return tier.discount
    return 0.0


def compute_tco(offering: DiscoverOffering, intent: BecknIntent) -> dict:
    q   = intent.quantity
    B   = intent.budget_constraints.max
    D   = intent.delivery_timeline

    delta   = volume_discount(offering, q)
    P_net   = offering.price_value * q * (1.0 - delta)
    tau     = D - offering.delivery_time_hours          # slack (hours)
    C_risk  = (ALPHA * offering.reliability_sigma * max(0.0, -tau)
               + BETA  * offering.reliability_sigma ** 2)
    tco     = P_net + offering.shipping_cost + C_risk + offering.ops_overhead

    feasible = (
        tau >= 0
        and tco <= B
        and not offering.blacklisted
    )
    reject_reason = []
    if tau < 0:       reject_reason.append(f"deadline_miss (τ={tau}h)")
    if tco > B:       reject_reason.append(f"over_budget (TCO=₹{tco:,.0f})")
    if offering.blacklisted: reject_reason.append("blacklisted")

    return {
        "offering_id":    offering.offering_id,
        "supplier_name":  offering.supplier_name,
        "delta":          delta,
        "P_net":          P_net,
        "C_risk":         C_risk,
        "tau":            tau,
        "tco":            tco,
        "feasible":       feasible,
        "reject_reason":  ", ".join(reject_reason) or "-",
    }


rows = [compute_tco(o, INTENT) for o in OFFERINGS]
df_raw = pd.DataFrame(rows)

print("=== Raw TCO breakdown ===")
display_cols = ["offering_id", "supplier_name", "delta", "P_net",
                "C_risk", "tau", "tco", "feasible", "reject_reason"]
print(df_raw[display_cols].to_string(index=False))

df_feasible = df_raw[df_raw["feasible"]].copy()
print(f"\n✓ Feasible set S_feasible: {list(df_feasible['offering_id'])} "
      f"({len(df_feasible)}/{len(df_raw)} offerings pass hard filter)")

## Cell 4 — Session-Level Min-Max Normalisation

Before the fixed-weight aggregator can combine heterogeneous raw signals (INR TCO, hours slack, dimensionless scores), each feature must be scaled to $[0, 1]$ **within the current session's feasible set**.

The normalisation is:

$$x_{i,k}^{\text{norm}} = \frac{x_{i,k} - \min_j x_{j,k}}{\max_j x_{j,k} - \min_j x_{j,k}}$$

with a zero-division guard: if $\max = \min$, all values receive $1.0$ (tie → neutral).

**Direction conventions** (higher normalised value = better):

| Raw feature | Direction | Normalised name |
|---|---|---|
| `tco` | ↓ lower is better → invert | `x_tco` |
| `tau` | ↑ more slack is better | `x_tau` |
| `C_risk` | ↓ lower is better → invert | `x_risk` |  
| qualitative score | ↑ higher is better (Cell 5) | `x_qual` |

In [ ]:
# Cell 4 — Min-max normalisation on the feasible set

def minmax_norm(series: pd.Series, invert: bool = False) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(1.0, index=series.index)
    normed = (series - lo) / (hi - lo)
    return 1.0 - normed if invert else normed


df_norm = df_feasible[["offering_id", "supplier_name",
                        "tco", "tau", "C_risk"]].copy()

# x_tco: inverted — lower TCO → higher score
df_norm["x_tco"]  = minmax_norm(df_feasible["tco"],    invert=True).values
# x_tau: direct  — more delivery slack → higher score
df_norm["x_tau"]  = minmax_norm(df_feasible["tau"],    invert=False).values
# x_risk: inverted — lower risk cost → higher score
df_norm["x_risk"] = minmax_norm(df_feasible["C_risk"], invert=True).values

# x_qual placeholder — will be filled in Cell 5
df_norm["x_qual"] = np.nan

print("=== Normalised quantitative features (feasible set) ===")
print(df_norm[["offering_id", "supplier_name",
               "tco", "x_tco",
               "tau", "x_tau",
               "C_risk", "x_risk"]].to_string(index=False))
print("\nx_qual will be filled by the ReAct agent in Cell 5.")

## Cell 5 — ReAct Agent Loop (LangGraph)

### Design

The agent follows the **ReAct pattern** (Yao et al., ICLR 2023): it alternates between a *Thought* (reasoning) step and an *Action* (tool call) step until it has gathered enough evidence to produce a `QualitativeScore`.

```
              ┌──────────────────────────────────────────────┐
              │              LangGraph StateGraph            │
              │                                              │
              │   react_agent_node                           │
              │        │                                     │
              │        ▼                                     │
              │   tool_router ──── search_supplier_history   │
              │        │       ├── fetch_esg_report          │
              │        │       └── check_certification       │
              │        │                                     │
              │        ▼ (observation)                       │
              │   react_agent_node  ←──── loop ──────────────┤
              │        │                                     │
              │        ▼ (finish_reason = STOP)              │
              │      END                                     │
              └──────────────────────────────────────────────┘
```

### QualitativeScore Schema

The agent emits a structured `QualitativeScore` for every feasible offering:

```python
class QualitativeScore(BaseModel):
    offering_id: str
    score: float        # ∈ [0, 1]
    confidence: float   # ∈ [0, 1]
    evidence: list[str] # short justification strings
```

**Mock implementation note:** All three tool functions return pre-programmed responses. No LLM API call is made — the agent's "reasoning" is scripted to demonstrate the intended flow.

In [ ]:
# Cell 5 — LangGraph ReAct agent with mocked tools
from typing import TypedDict

# ── Output schema ────────────────────────────────────────────────────────────

class QualitativeScore(BaseModel):
    offering_id: str
    score: float        = Field(ge=0.0, le=1.0)
    confidence: float   = Field(ge=0.0, le=1.0)
    evidence: list[str]


# ── Mock tool responses keyed by offering_id ─────────────────────────────────

_HISTORY_DB: dict[str, dict] = {
    "O1": {"on_time_rate": 0.92, "dispute_count": 1, "avg_delay_h": 1.2},
    "O2": {"on_time_rate": 0.78, "dispute_count": 4, "avg_delay_h": 5.6},
    "O3": {"on_time_rate": 0.97, "dispute_count": 0, "avg_delay_h": 0.4},
    "O5": {"on_time_rate": 0.99, "dispute_count": 0, "avg_delay_h": 0.1},
}
_ESG_DB: dict[str, dict] = {
    "O1": {"esg_score": 0.71, "carbon_rating": "B+", "labour_compliance": True},
    "O2": {"esg_score": 0.55, "carbon_rating": "C",  "labour_compliance": True},
    "O3": {"esg_score": 0.83, "carbon_rating": "A-", "labour_compliance": True},
    "O5": {"esg_score": 0.90, "carbon_rating": "A",  "labour_compliance": True},
}
_CERT_DB: dict[str, dict] = {
    "O1": {"iso_9001": True,  "bis_mark": True,  "expiry": "2027-03"},
    "O2": {"iso_9001": True,  "bis_mark": False, "expiry": "2026-09"},
    "O3": {"iso_9001": True,  "bis_mark": True,  "expiry": "2028-01"},
    "O5": {"iso_9001": True,  "bis_mark": True,  "expiry": "2027-11"},
}


@tool
def search_supplier_history(offering_id: str) -> dict:
    """Fetch historical delivery performance for a supplier offering."""
    return _HISTORY_DB.get(offering_id, {"error": "not_found"})

@tool
def fetch_esg_report(offering_id: str) -> dict:
    """Retrieve the latest ESG compliance report for a supplier offering."""
    return _ESG_DB.get(offering_id, {"error": "not_found"})

@tool
def check_certification(offering_id: str) -> dict:
    """Check ISO/BIS certification validity for a supplier offering."""
    return _CERT_DB.get(offering_id, {"error": "not_found"})


MOCK_TOOLS = {
    "search_supplier_history": search_supplier_history,
    "fetch_esg_report": fetch_esg_report,
    "check_certification": check_certification,
}


# ── Scoring heuristic (simulates GPT-4o structured output) ───────────────────

def _mock_llm_score(oid: str) -> QualitativeScore:
    """Simulate GPT-4o producing a QualitativeScore from gathered evidence.

    In production this is replaced by an instructor-patched Anthropic/OpenAI call:
        client.chat.completions.create(..., response_model=QualitativeScore)
    """
    hist  = _HISTORY_DB.get(oid, {})
    esg   = _ESG_DB.get(oid, {})
    cert  = _CERT_DB.get(oid, {})

    on_time   = hist.get("on_time_rate", 0.5)
    disputes  = hist.get("dispute_count", 5)
    esg_val   = esg.get("esg_score", 0.5)
    iso       = cert.get("iso_9001", False)
    bis       = cert.get("bis_mark", False)

    # Evidence-first scoring: each component weighted
    raw = (
        0.40 * on_time
        + 0.25 * esg_val
        + 0.20 * (1.0 if iso else 0.0)
        + 0.15 * (1.0 if bis else 0.0)
        - 0.05 * min(disputes, 5)
    )
    score = float(np.clip(raw, 0.0, 1.0))

    evidence = []
    evidence.append(f"on_time_rate={on_time:.0%}")
    if disputes > 2:
        evidence.append(f"WARNING: {disputes} disputes in last 12 months")
    evidence.append(f"ESG={esg_val:.2f} ({esg.get('carbon_rating','?')})")
    evidence.append(f"ISO9001={'✓' if iso else '✗'}  BIS={'✓' if bis else '✗'}")

    confidence = 0.90 if (hist and esg and cert) else 0.50
    return QualitativeScore(
        offering_id=oid, score=score,
        confidence=confidence, evidence=evidence
    )


# ── LangGraph agent definition ────────────────────────────────────────────────

class AgentState(TypedDict):
    offering_id: str
    messages: list
    tool_results: dict
    qualitative_score: Optional[QualitativeScore]


def react_agent_node(state: AgentState) -> AgentState:
    """Thought step: decide which tool to call next, or emit final score."""
    oid = state["offering_id"]
    gathered = state["tool_results"]

    # Simulate ReAct Thought → Action decision
    pending = [t for t in ["search_supplier_history",
                            "fetch_esg_report",
                            "check_certification"]
               if t not in gathered]

    if pending:
        next_tool = pending[0]
        thought = f"I need to call {next_tool} to assess {oid}."
        state["messages"].append(
            AIMessage(content=f"Thought: {thought}\nAction: {next_tool}({oid!r})")
        )
        # Execute tool synchronously in mock mode
        result = MOCK_TOOLS[next_tool].invoke({"offering_id": oid})
        gathered[next_tool] = result
        state["messages"].append(
            ToolMessage(content=json.dumps(result), tool_call_id=next_tool)
        )
    else:
        # All evidence gathered — produce structured output
        qs = _mock_llm_score(oid)
        state["qualitative_score"] = qs
        state["messages"].append(
            AIMessage(content=(
                f"Thought: All evidence gathered for {oid}. "
                f"Final score={qs.score:.3f}, confidence={qs.confidence:.2f}.\n"
                f"Evidence: {'; '.join(qs.evidence)}"
            ))
        )
    return state


def should_continue(state: AgentState) -> Literal["continue", "end"]:
    return "end" if state.get("qualitative_score") is not None else "continue"


def build_react_graph() -> StateGraph:
    g = StateGraph(AgentState)
    g.add_node("react_agent", react_agent_node)
    g.set_entry_point("react_agent")
    g.add_conditional_edges(
        "react_agent",
        should_continue,
        {"continue": "react_agent", "end": END},
    )
    return g.compile()


# ── Run the ReAct loop over every feasible offering ───────────────────────────

react_graph = build_react_graph()
qual_scores: list[QualitativeScore] = []

for oid in df_norm["offering_id"]:
    print(f"\n{'─'*60}")
    print(f"  ReAct loop → {oid}")
    print(f"{'─'*60}")

    init_state: AgentState = {
        "offering_id": oid,
        "messages": [HumanMessage(content=f"Score offering {oid} for procurement quality.")],
        "tool_results": {},
        "qualitative_score": None,
    }
    final_state = react_graph.invoke(init_state)

    qs = final_state["qualitative_score"]
    qual_scores.append(qs)

    for msg in final_state["messages"][1:]:   # skip the initial HumanMessage
        role = type(msg).__name__.replace("Message", "")
        print(f"  [{role}] {textwrap.shorten(msg.content, 120)}")

    print(f"  → QualitativeScore: {qs.score:.3f}  (confidence={qs.confidence:.2f})")
    print(f"     Evidence: {' | '.join(qs.evidence)}")

# Merge x_qual back into df_norm
qual_map = {qs.offering_id: qs.score for qs in qual_scores}
df_norm["x_qual"] = df_norm["offering_id"].map(qual_map)

print("\n✓ All qualitative scores populated.")

## Cell 6 — Fixed-Weight Aggregator & Final Ranking

### Scoring Formula

The four normalised signals are combined via a fixed linear aggregator:

$$S_i = 0.40 \cdot x_{\text{tco},i} + 0.25 \cdot x_{\tau,i} + 0.20 \cdot x_{\text{qual},i} + 0.15 \cdot x_{\text{risk},i}$$

| Weight | Feature | Rationale |
|--------|---------|----------|
| 0.40 | $x_{\text{tco}}$ | Cost dominates procurement decisions in Infosys policy |
| 0.25 | $x_{\tau}$ | Delivery slack directly maps to SLA compliance risk |
| 0.20 | $x_{\text{qual}}$ | LLM-derived qualitative signal — high leverage but costly |
| 0.15 | $x_{\text{risk}}$ | Reliability variance hedges against tail-risk delays |

The offering with $\arg\max_i S_i$ is returned as `selected` in the `POST /score` response.

> **Phase 1 limitation:** these weights are hand-coded domain priors. They cannot adapt to category-specific or buyer-specific preferences without manual retuning — the primary motivation for the Phase 2 Learning-to-Rank upgrade. See `phase2_learning_to_rank.md`.

In [ ]:
# Cell 6 — Fixed-weight aggregation, ranking, and winner selection

W_TCO  = 0.40
W_TAU  = 0.25
W_QUAL = 0.20
W_RISK = 0.15
assert abs(W_TCO + W_TAU + W_QUAL + W_RISK - 1.0) < 1e-9, "Weights must sum to 1.0"

df_norm["S"] = (
    W_TCO  * df_norm["x_tco"]
    + W_TAU  * df_norm["x_tau"]
    + W_QUAL * df_norm["x_qual"]
    + W_RISK * df_norm["x_risk"]
)

df_ranked = (
    df_norm
    .sort_values("S", ascending=False)
    .reset_index(drop=True)
)
df_ranked.index += 1   # 1-based rank
df_ranked.index.name = "rank"

print("=" * 70)
print("FINAL RANKING — Phase 1 Hybrid ReAct Pipeline")
print("=" * 70)
print(df_ranked[["offering_id", "supplier_name",
                  "x_tco", "x_tau", "x_qual", "x_risk", "S"]].to_string())

winner_row = df_ranked.iloc[0]
winner_oid = winner_row["offering_id"]
winner_obj = next(o for o in OFFERINGS if o.offering_id == winner_oid)

print()
print("=" * 70)
print(f"  SELECTED OFFERING: {winner_oid} — {winner_obj.supplier_name}")
print(f"  Composite score S = {winner_row['S']:.4f}")
print(f"  Unit price        = ₹{winner_obj.price_value:,.2f}")
print(f"  Delivery          = {winner_obj.delivery_time_hours} h"
      f"  (slack τ = {winner_row['tau']:.0f} h)")

# Retrieve qualitative evidence for the winner
winner_qs = next(qs for qs in qual_scores if qs.offering_id == winner_oid)
print(f"  Qualitative score = {winner_qs.score:.3f}"
      f"  (confidence={winner_qs.confidence:.2f})")
print(f"  Evidence          : {' | '.join(winner_qs.evidence)}")
print("=" * 70)

# Simulate /score HTTP response body
response_payload = {
    "selected": winner_obj.model_dump(),
    "_meta": {
        "pipeline": "phase1_hybrid_react",
        "feasible_count": len(df_feasible),
        "total_count": len(OFFERINGS),
        "composite_score": round(float(winner_row["S"]), 4),
        "weights": {"tco": W_TCO, "tau": W_TAU, "qual": W_QUAL, "risk": W_RISK},
    }
}
print("\n/score response payload (excerpt):")
print(json.dumps(response_payload["_meta"], indent=2))
print("\n✓ Phase 1 pipeline complete. Selected offering forwarded to POST /score.")